# Qwen3-ASR-0.6B — LoRA fine-tuning (Arabic)

Sibling of `asr_model_agnostic_finetune.ipynb` (OmniASR), same pipeline shape
(ConfigAPI → ModelAdapter → PredictAPI → EvaluateAPI → TrainAPI) but for
**`Qwen/Qwen3-ASR-0.6B-hf`**, which is **transformers-native** (not fairseq2).

### Key differences from the OmniASR notebook
1. **Instruction / chat model.** Training goes through
   `processor.apply_chat_template(chat, tokenize=True, return_dict=True, output_labels=True)`
   — audio + transcript in one user turn — then `loss = model(**inputs).loss`. This chat-template
   data prep is **mandatory**, it is the model's only supported training path.
2. **Real PEFT LoRA.** The Qwen3 decoder projections are ordinary `torch.nn.Linear`
   (`q/k/v/o_proj`, `gate/up/down_proj`), so standard PEFT `get_peft_model` wraps them and
   `save_pretrained` / adapter reload round-trip natively — none of OmniASR's manual `_LoRALinear`
   injection is needed.
3. **Separate venv.** Qwen3-ASR needs transformers >= 5.13 (env: `/workspace/venv_qwen_gpu`,
   transformers 5.14.1) which conflicts with OmniASR's fairseq2-pinned transformers 4.57.6.
4. **Inference** via `processor.apply_transcription_request(audio=..., language="ar")` →
   `model.generate` → `processor.decode(..., return_format="transcription_only")`.


## Cell 1 — Environment

In [1]:
# Env is prebuilt at /workspace/venv_qwen_gpu (transformers 5.14.1 + peft + torch cu128).
# To rebuild elsewhere:
# !pip install "transformers>=5.13" peft accelerate datasets jiwer soundfile librosa bitsandbytes
import os, json, gc, math, time, random, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Callable

import numpy as np, torch, torch.nn as nn
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
print(torch.__version__, torch.cuda.is_available())

2.8.0+cu128 True


## Cell 2 — Config knobs

In [2]:
MODEL_NAME = "Qwen/Qwen3-ASR-0.6B-hf"

LANG          = "ar"                # Qwen3-ASR language hint (ISO code)
SMOKE_TEST    = True                # tiny subsets + 2 epochs
SEED          = 42

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ROOT          = Path(os.environ.get("ASR_ENV_ROOT", "/workspace/asr_env"))
MODEL_CACHE   = ROOT / "models"
PRED_DIR      = ROOT / "preds"
METRIC_DIR    = ROOT / "metrics"
CKPT_DIR      = ROOT / "checkpoints"
for d in (MODEL_CACHE, PRED_DIR, METRIC_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
os.environ["WANDB_PROJECT"] = "arabic-asr-qwen3"
os.environ.setdefault("WANDB_MODE", "disabled")   # no W&B account on this box; per-epoch prints suffice

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print(f"DEVICE={DEVICE} | dtype={COMPUTE_DTYPE} | ROOT={ROOT}")

DEVICE=cuda | dtype=torch.bfloat16 | ROOT=/workspace/asr_env


## Cell 3 — Arabic normalization + WER/CER (identical to the OmniASR notebook)

In [3]:
import re, unicodedata, jiwer

_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
_PUNC = re.compile(r"[^\w\s\u0621-\u064A]")

def normalize_ar(t: str) -> str:
    """Diacritic strip, tatweel removal, alef/ya/ta-marbuta unification."""
    if t is None: return ""
    t = unicodedata.normalize("NFKC", str(t))
    t = _DIAC.sub("", t)
    t = re.sub("[\u0622\u0623\u0625\u0671]", "\u0627", t)   # alef variants -> alef
    t = t.replace("\u0649", "\u064A")                          # alef maqsura -> ya
    t = t.replace("\u0629", "\u0647")                          # ta marbuta -> ha
    t = t.replace("\u0624", "\u0648").replace("\u0626", "\u064A")
    t = _PUNC.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def compute_wer_cer(preds, refs, normalize=True):
    if normalize:
        preds = [normalize_ar(p) for p in preds]
        refs  = [normalize_ar(r) for r in refs]
    keep = [(p, r) for p, r in zip(preds, refs) if r.strip()]
    if not keep: return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    p, r = zip(*keep)
    return {"wer": jiwer.wer(list(r), list(p)),
            "cer": jiwer.cer(list(r), list(p)),
            "n": len(r)}

## Cell 4 — ConfigAPI

Qwen3-ASR LoRA targets are the standard Qwen3 decoder projections (all real `nn.Linear`).
`max_audio_seconds=30` matches Qwen3-ASR's native feature-extractor chunk (n_samples=480000 = 30s
@ 16 kHz).

In [4]:
@dataclass
class LoRAConfigSpec:
    r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    # Qwen3 decoder attention + MLP projections. These leaf names also appear in the audio
    # tower (q/k/v_proj), so PEFT adapts the text decoder fully + audio-encoder attention -- a
    # standard, effective ASR-LoRA footprint.
    target_modules: List[str] = field(default_factory=lambda: [
        "q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    modules_to_save: Optional[List[str]] = None
    task_type: Optional[str] = None      # custom multimodal head -> leave None (LoRA still injects)

@dataclass
class TrainConfigSpec:
    num_epochs: int = 50
    early_stopping_patience: int = 4
    metric_for_best: str = "wer"
    greater_is_better: bool = False
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    optim: str = "adamw_bnb_8bit"
    bf16: bool = True
    gradient_checkpointing: bool = False   # not wired (PEFT+grad-ckpt needs enable_input_require_grads)
    dataloader_num_workers: int = 0        # 0 avoids fork+CUDA issues when collate runs the processor
    max_audio_seconds: float = 30.0        # Qwen3-ASR native chunk length
    max_label_tokens: int = 256
    save_total_limit: int = 2

class ConfigAPI:
    _LORA = {"Qwen/Qwen3-ASR-0.6B-hf": LoRAConfigSpec()}
    _TRAIN = {"Qwen/Qwen3-ASR-0.6B-hf": TrainConfigSpec()}
    @classmethod
    def lora(cls, name)  -> LoRAConfigSpec:  return cls._LORA.get(name, LoRAConfigSpec())
    @classmethod
    def train(cls, name) -> TrainConfigSpec: return cls._TRAIN.get(name, TrainConfigSpec())

print(json.dumps(asdict(ConfigAPI.lora(MODEL_NAME)), indent=2))
print(json.dumps(asdict(ConfigAPI.train(MODEL_NAME)), indent=2))

{
  "r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "bias": "none",
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "modules_to_save": null,
  "task_type": null
}
{
  "num_epochs": 50,
  "early_stopping_patience": 4,
  "metric_for_best": "wer",
  "greater_is_better": false,
  "per_device_train_batch_size": 2,
  "per_device_eval_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.0,
  "max_grad_norm": 1.0,
  "optim": "adamw_bnb_8bit",
  "bf16": true,
  "gradient_checkpointing": false,
  "dataloader_num_workers": 0,
  "max_audio_seconds": 30.0,
  "max_label_tokens": 256,
  "save_total_limit": 2
}


## Cell 5 — ModelAdapter ABC

In [5]:
from abc import ABC, abstractmethod

class ModelAdapter(ABC):
    name: str
    loss_type: str = "seq2seq"
    supports_unsloth: bool = False

    def __init__(self, model_name: str, lang: str = LANG):
        self.model_name = model_name; self.lang = lang
        self.model = None; self.processor = None

    @abstractmethod
    def load_base(self): ...
    @abstractmethod
    def preprocess(self, example: Dict) -> Dict: ...
    @abstractmethod
    def collate(self, features: List[Dict]) -> Dict[str, Any]: ...
    @abstractmethod
    def generate(self, batch: Dict) -> List[str]: ...

    def apply_lora(self, spec: "LoRAConfigSpec"):
        """PEFT LoRA. Qwen3-ASR projections are torch.nn.Linear, so this Just Works."""
        from peft import LoraConfig, get_peft_model
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        self.model = get_peft_model(self.model, LoraConfig(**kw))
        self.model.print_trainable_parameters()
        print("[lora] peft"); return self.model

## Cell 6 — Qwen3-ASR adapter

In [6]:
class Qwen3ASRAdapter(ModelAdapter):
    """Qwen/Qwen3-ASR-0.6B-hf (transformers-native). Verified on GPU (2026-07-16):
      * load:   AutoProcessor + Qwen3ASRForConditionalGeneration (bf16)
      * TRAIN:  processor.apply_chat_template(chat, output_labels=True) -> loss = model(**in).loss
                chat = user turn holding {type:text, text:transcript} + {type:audio, audio:ndarray}
      * INFER:  processor.apply_transcription_request(audio=[...], language=[...]) -> model.generate
                -> processor.decode(gen, return_format="transcription_only")
      * inputs MUST be cast to the model dtype (BatchFeature.to(device, dtype) casts float only).
      * LoRA:   real PEFT on q/k/v/o_proj + gate/up/down_proj.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    def load_base(self):
        from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration
        print(f"[load] {self.model_name}")
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = Qwen3ASRForConditionalGeneration.from_pretrained(
            self.model_name, dtype=COMPUTE_DTYPE).to(DEVICE)
        self.model.config.use_cache = False
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def collate(self, feats):
        chat = [[{"role": "user", "content": [
                    {"type": "text",  "text":  f["text"]},
                    {"type": "audio", "audio": f["audio"]}]}]
                for f in feats]
        inputs = self.processor.apply_chat_template(
            chat, tokenize=True, return_dict=True, output_labels=True)
        batch = dict(inputs)                       # BatchFeature -> plain dict of tensors
        batch["text"]   = [f["text"] for f in feats]
        batch["_audio"] = [f["audio"] for f in feats]
        return batch

    _MODEL_KEYS = ("input_ids", "attention_mask", "input_features", "input_features_mask", "labels")

    def train_step(self, batch) -> torch.Tensor:
        inputs = {}
        for k in self._MODEL_KEYS:
            v = batch.get(k)
            if v is None: continue
            v = v.to(DEVICE)
            if v.is_floating_point(): v = v.to(COMPUTE_DTYPE)   # input_features -> bf16
            inputs[k] = v
        return self.model(**inputs).loss

    @torch.no_grad()
    def generate(self, batch):
        audios = list(batch["_audio"])
        req = self.processor.apply_transcription_request(
            audio=audios, language=[self.lang] * len(audios))
        req = req.to(DEVICE, COMPUTE_DTYPE)
        out_ids = self.model.generate(**req, max_new_tokens=256)
        gen = out_ids[:, req["input_ids"].shape[1]:]
        return [str(t) for t in self.processor.decode(gen, return_format="transcription_only")]

## Cell 7 — Registry

In [7]:
REGISTRY: Dict[str, Callable[..., ModelAdapter]] = {
    "Qwen/Qwen3-ASR-0.6B-hf": Qwen3ASRAdapter,
    # "Qwen/Qwen3-ASR-1.7B-hf": Qwen3ASRAdapter,   # same adapter, bigger checkpoint
}

def get_adapter(name, **kw) -> ModelAdapter:
    if name not in REGISTRY: raise KeyError(f"{name} not registered. Have: {list(REGISTRY)}")
    a = REGISTRY[name](name, **kw); a.name = name; return a

## Cell 8 — Datasets

Same real North-Levantine (Palestinian) Arabic corpus + soundfile decode as the OmniASR notebook.
Smoke test uses clips <= 30s (Qwen3-ASR's native chunk length).

In [8]:
from datasets import load_from_disk, Audio, Dataset
import soundfile as sf, io, random as _random

REAL_DATA_DIR = Path("/workspace/asr/Palestinian-ASR/omnilingual_selected/apc_north_levantine_all_splits")

def _materialize_audio(ds):
    def gen():
        for row in ds:
            wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            out = dict(row); out["audio"] = {"array": wav, "sampling_rate": sr}
            yield out
    return Dataset.from_generator(gen)

def load_splits(smoke=SMOKE_TEST):
    ds = load_from_disk(str(REAL_DATA_DIR))
    if "raw_text" in ds.column_names and "text" not in ds.column_names:
        ds = ds.rename_column("raw_text", "text")
    ds = ds.cast_column("audio", Audio(decode=False))
    if smoke:
        durations = ds["duration"]
        short_idx = [i for i, d in enumerate(durations) if d <= 30.0]
        _random.Random(SEED).shuffle(short_idx)
        # 1 fake train / 1 fake val / 1 fake test sample -- real audio+text.
        splits = {"train": ds.select(short_idx[0:1]),
                  "validation": ds.select(short_idx[1:2]),
                  "test": ds.select(short_idx[2:3])}
    else:
        ds = ds.shuffle(seed=SEED)
        n = len(ds); n_tr, n_va = int(n * 0.8), int(n * 0.9)
        splits = {"train": ds.select(range(0, n_tr)),
                  "validation": ds.select(range(n_tr, n_va)),
                  "test": ds.select(range(n_va, n))}
    for k in splits: splits[k] = _materialize_audio(splits[k])
    return splits

SPLITS = load_splits()
{k: len(v) for k, v in SPLITS.items()}

{'train': 1, 'validation': 1, 'test': 1}

## Cell 9 — PredictAPI (cached)

In [9]:
def _fingerprint(ds) -> str:
    try: h = ds._fingerprint
    except Exception: h = str(len(ds))
    return hashlib.md5(f"{h}{len(ds)}".encode()).hexdigest()[:10]

class PredictAPI:
    @staticmethod
    def _path(model_name, split, ds, stage):
        slug = model_name.replace("/", "__")
        return PRED_DIR / f"{slug}__{split}__{_fingerprint(ds)}__{stage}.json"

    @staticmethod
    def run(adapter, ds, split="test", stage="base", batch_size=4, force=False):
        p = PredictAPI._path(adapter.name, split, ds, stage)
        if p.exists() and not force:
            print(f"[predict] CACHE HIT -> {p.name}"); return json.loads(p.read_text(encoding="utf-8"))
        print(f"[predict] generating ({stage}, {split}, n={len(ds)})")
        feats = [adapter.preprocess(ex) for ex in ds]
        preds, refs = [], []
        adapter.model.eval()
        for i in range(0, len(feats), batch_size):
            b = adapter.collate(feats[i:i+batch_size])
            b_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            preds.extend(adapter.generate(b_dev)); refs.extend(b["text"])
            print(f"  {min(i+batch_size,len(feats))}/{len(feats)}", end="\r")
        rec = {"model": adapter.name, "split": split, "stage": stage,
               "predictions": preds, "references": refs, "n": len(preds), "ts": time.time()}
        p.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n[predict] saved -> {p.name}")
        return rec

## Cell 10 — EvaluateAPI (cached)

In [10]:
class EvaluateAPI:
    @staticmethod
    def _path(model_name, split, stage):
        return METRIC_DIR / f"{model_name.replace('/','__')}__{split}__{stage}.json"

    @staticmethod
    def run(model_name, pred_record, split="test", stage="base", force=False):
        p = EvaluateAPI._path(model_name, split, stage)
        if p.exists() and not force:
            m = json.loads(p.read_text()); print(f"[eval] CACHE HIT -> {m}"); return m
        m = compute_wer_cer(pred_record["predictions"], pred_record["references"])
        m.update({"model": model_name, "split": split, "stage": stage})
        p.write_text(json.dumps(m, indent=2))
        print(f"[eval] WER={m['wer']:.4f} CER={m['cer']:.4f} (n={m['n']}) -> {p.name}")
        return m

## Cell 11 — Build adapter + load base model

In [11]:
set_seed()
adapter = get_adapter(MODEL_NAME, lang=LANG)
adapter.load_base()
n_params = sum(p.numel() for p in adapter.model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M params | loss_type={adapter.loss_type}")

[load] Qwen/Qwen3-ASR-0.6B-hf


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 30/611 [00:00<00:02, 249.84it/s]

Loading weights:  16%|█▌        | 96/611 [00:00<00:01, 433.31it/s]

Loading weights:  26%|██▌       | 160/611 [00:00<00:00, 496.43it/s]

...

Loading weights:  76%|███████▋  | 467/611 [00:00<00:00, 462.83it/s]

Loading weights:  84%|████████▍ | 515/611 [00:01<00:00, 454.73it/s]

Loading weights:  92%|█████████▏| 562/611 [00:01<00:00, 401.88it/s]

Loading weights: 100%|██████████| 611/611 [00:01<00:00, 470.36it/s]

Qwen/Qwen3-ASR-0.6B-hf: 782.4M params | loss_type=seq2seq


## Cell 12 — Baseline preds + eval on test (cached)

In [12]:
base_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="base",
                              batch_size=ConfigAPI.train(MODEL_NAME).per_device_eval_batch_size)
base_metrics = EvaluateAPI.run(MODEL_NAME, base_preds, split="test", stage="base")
for p, r in list(zip(base_preds["predictions"], base_preds["references"]))[:3]:
    print(f"REF : {r}\nHYP : {p}\n")

[predict] generating (base, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__83667d504c__base.json
[eval] WER=0.5135 CER=0.2367 (n=1) -> Qwen__Qwen3-ASR-0.6B-hf__test__base.json
REF : hesitation فيا مدينه اسطنبول الي هي بتكون بالنص بين اسيا وافريييا وبين بين اسيا واوروبا الي بيناتن بحر noise الي هي ماضيق البوسفور فا تركيا بل تركيا بالويت الحالي هي hesitation متطوره بلد مزدهر وبلد كتير حلو
HYP : في مدينة طنبو اللي هي بتكون بالنص بين آسيا وأفريقيا وبين بين آسيا وأوروبا. لبيناتن بحر اللي هي مديق البصفور. فتركيا بالتركيا بالوقت الحالي هي مطورة بلد مزدهر وبلد كثير حلو.



## Cell 13 — Apply LoRA

In [13]:
lora_spec  = ConfigAPI.lora(MODEL_NAME)
train_spec = ConfigAPI.train(MODEL_NAME)
if SMOKE_TEST:
    train_spec.num_epochs = 2
    train_spec.early_stopping_patience = 4
adapter.apply_lora(lora_spec)

trainable params: 23,281,664 || all params: 805,707,776 || trainable%: 2.8896
[lora] peft


PeftModel(
  (base_model): LoraModel(
    (model): Qwen3ASRForConditionalGeneration(
      (model): Qwen3ASRModel(
        (audio_tower): Qwen3ASREncoder(
          (positional_embedding): SinusoidsPositionEmbedding()
          (layers): ModuleList(
            (0-17): 18 x Qwen3ASRAudioEncoderLayer(
              (self_attn): Qwen3ASRAudioAttention(
                (k_proj): lora.Linear(
                  (base_layer): Linear(in_features=896, out_features=896, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=896, out_features=32, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=32, out_features=896, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()

## Cell 14 — TrainAPI

Same custom loop as the OmniASR notebook (per-epoch train loss / val loss / val WER / val CER,
early stopping on WER patience 4, best-WER checkpoint). Two Qwen-specific points:
`_validate` passes the full batch to `train_step` (which selects its own keys), and the best
checkpoint is a real PEFT adapter saved via `save_pretrained`.

In [14]:
import wandb
from contextlib import nullcontext
from torch.utils.data import DataLoader

def _amp(spec):
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

class _ListDS(torch.utils.data.Dataset):
    def __init__(self, feats): self.f = feats
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i]

class TrainAPI:
    @staticmethod
    def _prep(adapter, ds, spec):
        feats, dropped = [], 0
        for ex in ds:
            f = adapter.preprocess(ex)
            if f["audio_len"] > spec.max_audio_seconds: dropped += 1; continue
            if not f["text"].strip():                   dropped += 1; continue
            feats.append(f)
        print(f"[prep] kept {len(feats)}, dropped {dropped}")
        return feats

    @staticmethod
    @torch.no_grad()
    def _validate(adapter, loader, spec):
        adapter.model.eval(); losses, preds, refs = [], [], []
        for b in loader:
            g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            try:
                with _amp(spec):
                    losses.append(float(adapter.train_step(g)))
            except Exception as e:
                print(f"[val] loss skipped: {e}")
            preds.extend(adapter.generate(g)); refs.extend(b["text"])
        m = compute_wer_cer(preds, refs)
        m["val_loss"] = float(np.mean(losses)) if losses else float("nan")
        return m

    @staticmethod
    def run(adapter, splits, spec, lora_spec):
        slug = adapter.name.replace("/", "__")
        run  = wandb.init(project=os.environ["WANDB_PROJECT"], name=f"{slug}-lora",
                          config={**asdict(spec), **asdict(lora_spec),
                                  "model": adapter.name, "lang": LANG, "smoke": SMOKE_TEST},
                          reinit=True)
        best_dir = CKPT_DIR / slug / "best"; best_dir.mkdir(parents=True, exist_ok=True)

        tr_f = TrainAPI._prep(adapter, splits["train"], spec)
        va_f = TrainAPI._prep(adapter, splits["validation"], spec)
        tr = DataLoader(_ListDS(tr_f), batch_size=spec.per_device_train_batch_size, shuffle=True,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers,
                        pin_memory=False, drop_last=False)
        va = DataLoader(_ListDS(va_f), batch_size=spec.per_device_eval_batch_size, shuffle=False,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers)

        params = [p for p in adapter.model.parameters() if p.requires_grad]
        opt = None
        if DEVICE == "cuda":
            try:
                import bitsandbytes as bnb
                opt = bnb.optim.AdamW8bit(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)
            except Exception as e:
                print(f"[opt] AdamW8bit unavailable ({e}); using torch.AdamW")
        if opt is None:
            opt = torch.optim.AdamW(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)

        steps_pe = max(1, math.ceil(len(tr) / spec.gradient_accumulation_steps))
        total    = steps_pe * spec.num_epochs
        from transformers import get_linear_schedule_with_warmup
        sched = get_linear_schedule_with_warmup(opt, int(total*spec.warmup_ratio), total)

        best_wer, bad_epochs, gstep, history = float("inf"), 0, 0, []
        for epoch in range(1, spec.num_epochs + 1):
            adapter.model.train(); ep_loss, nb = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, b in enumerate(tr):
                g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
                with _amp(spec):
                    loss = adapter.train_step(g) / spec.gradient_accumulation_steps
                if not torch.isfinite(loss):
                    print(f"[nan] step {i} non-finite loss, skipping"); opt.zero_grad(set_to_none=True); continue
                loss.backward()
                if (i + 1) % spec.gradient_accumulation_steps == 0 or (i + 1) == len(tr):
                    gnorm = torch.nn.utils.clip_grad_norm_(params, spec.max_grad_norm)
                    if not torch.isfinite(gnorm):
                        print(f"[nan] step {i} non-finite grad norm, skipping"); opt.zero_grad(set_to_none=True); continue
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                    wandb.log({"train/step_loss": float(loss)*spec.gradient_accumulation_steps,
                               "train/grad_norm": float(gnorm), "train/lr": sched.get_last_lr()[0]}, step=gstep)
                ep_loss += float(loss) * spec.gradient_accumulation_steps; nb += 1

            train_loss = ep_loss / max(nb, 1)
            vm = TrainAPI._validate(adapter, va, spec)
            row = {"epoch": epoch, "train_loss": train_loss, "val_loss": vm["val_loss"],
                   "val_wer": vm["wer"], "val_cer": vm["cer"]}
            history.append(row)
            wandb.log({"epoch": epoch, "train/loss": train_loss, "val/loss": vm["val_loss"],
                       "val/wer": vm["wer"], "val/cer": vm["cer"]}, step=gstep)
            print(f"epoch {epoch:>3} | train {train_loss:.4f} | val {vm['val_loss']:.4f} "
                  f"| WER {vm['wer']:.4f} | CER {vm['cer']:.4f}")

            if vm["wer"] < best_wer - 1e-6:
                best_wer, bad_epochs = vm["wer"], 0
                adapter.model.save_pretrained(str(best_dir))     # real PEFT adapter save
                (best_dir / "best.json").write_text(json.dumps({**row, "gstep": gstep}, indent=2))
                print(f"  -> new best WER {best_wer:.4f}, saved to {best_dir}")
            else:
                bad_epochs += 1
                print(f"  -> no improvement ({bad_epochs}/{spec.early_stopping_patience})")
                if bad_epochs >= spec.early_stopping_patience:
                    print(f"[early-stop] epoch {epoch}, best WER {best_wer:.4f}"); break

        wandb.summary["best_val_wer"] = best_wer
        (CKPT_DIR / slug / "history.json").write_text(json.dumps(history, indent=2))
        return {"best_wer": best_wer, "best_dir": str(best_dir), "history": history, "run": run}

## Cell 15 — Train

In [15]:
set_seed()
train_out = TrainAPI.run(adapter, SPLITS, train_spec, lora_spec)
print(f"best val WER: {train_out['best_wer']:.4f} @ {train_out['best_dir']}")

[prep] kept 1, dropped 0


[prep] kept 1, dropped 0


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


epoch   1 | train 20.1788 | val 15.8421 | WER 0.4035 | CER 0.1604


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  -> new best WER 0.4035, saved to /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


epoch   2 | train 16.1340 | val 13.1693 | WER 0.3860 | CER 0.1536


  -> new best WER 0.3860, saved to /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best
best val WER: 0.3860 @ /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best


## Cell 16 — Load best checkpoint, predict + evaluate on test, save

In [16]:
best = Path(train_out["best_dir"])
# Restore the best PEFT adapter weights into the live model (in-memory weights are the LAST
# epoch, not necessarily the best). set_peft_model_state_dict is the robust PEFT idiom.
try:
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    sd = load_file(str(best / "adapter_model.safetensors"))
    set_peft_model_state_dict(adapter.model, sd)
    print(f"[ckpt] restored best PEFT adapter <- {best}")
except Exception as e:
    print(f"[ckpt] safetensors restore failed ({e}); trying load_adapter")
    adapter.model.load_adapter(str(best), adapter_name="default")

tuned_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="tuned",
                               batch_size=train_spec.per_device_eval_batch_size, force=True)
tuned_metrics = EvaluateAPI.run(MODEL_NAME, tuned_preds, split="test", stage="tuned", force=True)

summary = {
    "model": MODEL_NAME, "lang": LANG, "smoke_test": SMOKE_TEST,
    "base":  {"wer": base_metrics["wer"],  "cer": base_metrics["cer"]},
    "tuned": {"wer": tuned_metrics["wer"], "cer": tuned_metrics["cer"]},
    "delta": {"wer": base_metrics["wer"] - tuned_metrics["wer"],
              "cer": base_metrics["cer"] - tuned_metrics["cer"]},
    "best_val_wer": train_out["best_wer"],
    "lora": asdict(lora_spec), "train": asdict(train_spec),
}
sp = METRIC_DIR / f"{MODEL_NAME.replace('/','__')}__SUMMARY.json"
sp.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

wandb.log({"test/base_wer": base_metrics["wer"],   "test/base_cer": base_metrics["cer"],
           "test/tuned_wer": tuned_metrics["wer"], "test/tuned_cer": tuned_metrics["cer"]})
wandb.finish()
print(json.dumps(summary, indent=2, ensure_ascii=False))

[ckpt] restored best PEFT adapter <- /workspace/asr_env/checkpoints/Qwen__Qwen3-ASR-0.6B-hf/best
[predict] generating (tuned, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__83667d504c__tuned.json
[eval] WER=0.5946 CER=0.2609 (n=1) -> Qwen__Qwen3-ASR-0.6B-hf__test__tuned.json
{
  "model": "Qwen/Qwen3-ASR-0.6B-hf",
  "lang": "ar",
  "smoke_test": true,
  "base": {
    "wer": 0.5135135135135135,
    "cer": 0.23671497584541062
  },
  "tuned": {
    "wer": 0.5945945945945946,
    "cer": 0.2608695652173913
  },
  "delta": {
    "wer": -0.08108108108108114,
    "cer": -0.024154589371980673
  },
  "best_val_wer": 0.38596491228070173,
  "lora": {
    "r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "modules_to_save": null,
    "task_type": null
  },
  "train": {
    "num_epochs": 2,
    "early_stopping_patience": 4,
    "metric_for_best": "wer",
    "greater_is_better": false,
    "per_device_train_batch_size": 2,
    "per_device_e

## Cell 17 — One-shot smoke wrapper

Same code path end-to-end (load → base predict/eval → LoRA → train → tuned predict/eval),
callable for any registered Qwen3-ASR checkpoint.

In [17]:
def smoke(model_name, splits):
    set_seed()
    a = get_adapter(model_name, lang=LANG); a.load_base()
    bp = PredictAPI.run(a, splits["test"], "test", "base", batch_size=2)
    bm = EvaluateAPI.run(model_name, bp, "test", "base")
    ls, ts = ConfigAPI.lora(model_name), ConfigAPI.train(model_name)
    ts.num_epochs = 2; ts.per_device_train_batch_size = 1; ts.gradient_accumulation_steps = 2
    a.apply_lora(ls)
    out = TrainAPI.run(a, splits, ts, ls)
    tp = PredictAPI.run(a, splits["test"], "test", "tuned", batch_size=2, force=True)
    tm = EvaluateAPI.run(model_name, tp, "test", "tuned", force=True)
    del a.model, a; gc.collect(); torch.cuda.empty_cache()
    return {"model": model_name, "base_wer": bm["wer"], "tuned_wer": tm["wer"],
            "best_val_wer": out["best_wer"], "status": "PASS"}

results = []
for m in ["Qwen/Qwen3-ASR-0.6B-hf"]:
    try:
        results.append(smoke(m, SPLITS))
    except Exception as e:
        import traceback; traceback.print_exc()
        results.append({"model": m, "status": f"FAIL: {e}"})
    print("=" * 70)

import pandas as pd
pd.DataFrame(results)

[load] Qwen/Qwen3-ASR-0.6B-hf


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

Loading weights:  65%|██████▌   | 400/611 [00:00<00:00, 3984.52it/s]

Loading weights: 100%|██████████| 611/611 [00:00<00:00, 3816.62it/s]

[predict] CACHE HIT -> Qwen__Qwen3-ASR-0.6B-hf__test__83667d504c__base.json
[eval] CACHE HIT -> {'wer': 0.5135135135135135, 'cer': 0.23671497584541062, 'n': 1, 'model': 'Qwen/Qwen3-ASR-0.6B-hf', 'split': 'test', 'stage': 'base'}


...

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


epoch   2 | train 16.1420 | val 13.1596 | WER 0.4035 | CER 0.1604
  -> no improvement (1/4)
[predict] generating (tuned, test, n=1)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  1/1
[predict] saved -> Qwen__Qwen3-ASR-0.6B-hf__test__83667d504c__tuned.json
[eval] WER=0.5946 CER=0.2609 (n=1) -> Qwen__Qwen3-ASR-0.6B-hf__test__tuned.json


,model,base_wer,tuned_wer,best_val_wer,status
0,Qwen/Qwen3-ASR-0.6B-hf,0.513514,0.594595,0.385965,PASS
